# chematic quickstart

**Replace RDKit in 10 minutes** — pure-Rust cheminformatics, zero C/C++ dependencies.

[![PyPI](https://img.shields.io/pypi/v/chematic.svg)](https://pypi.org/project/chematic/)
[![GitHub](https://img.shields.io/badge/github-kent--tokyo%2Fchematic-blue)](https://github.com/kent-tokyo/chematic)

This notebook covers the most common cheminformatics tasks:
1. Parsing molecules and computing descriptors
2. Drug-likeness filters (Lipinski, PAINS, pKa, ADMET)
3. Fingerprints and similarity
4. Bulk parallel processing
5. Substructure search (SMARTS)
6. Molecular transformations
7. 2D visualization
8. Virtual screening (shape similarity)

In [ ]:
# Install chematic (no conda, no C compiler required)
%pip install chematic pandas -q

## 1. Parse a molecule and compute basic descriptors

In [ ]:
import chematic

mol = chematic.from_smiles("CC(=O)Oc1ccccc1C(=O)O")  # aspirin

print(f"Formula : {mol.formula}")
print(f"MW      : {mol.mw:.2f} Da")
print(f"LogP    : {mol.logp:.2f}")
print(f"TPSA    : {mol.tpsa:.1f} Å²")
print(f"HBD/HBA : {mol.hbd} / {mol.hba}")
print(f"QED     : {mol.qed:.3f}")
print(f"InChIKey: {mol.inchikey}")

## 2. Drug-likeness filters, pKa, and ADMET

In [ ]:
print("--- Drug-likeness filters ---")
print(f"Lipinski : {mol.lipinski_passes}")
print(f"Veber    : {mol.veber_passes}")
print(f"PAINS    : {mol.pains_passes}")
print(f"Brenk    : {mol.brenk_passes}")

print("\n--- pKa ---")
pka = mol.pka()
print(f"Most acidic : {pka['most_acidic']}")
print(f"Most basic  : {pka['most_basic']}")

print("\n--- ADMET ---")
admet = mol.admet()
for k, v in admet.items():
    print(f"  {k:20s}: {v}")

## 3. Fingerprints and Tanimoto similarity

In [ ]:
aspirin   = chematic.from_smiles("CC(=O)Oc1ccccc1C(=O)O")
ibuprofen = chematic.from_smiles("CC(C)Cc1ccc(CC(C)C(=O)O)cc1")
paracetamol = chematic.from_smiles("CC(=O)Nc1ccc(O)cc1")

fp_a = aspirin.ecfp4()
fp_i = ibuprofen.ecfp4()
fp_p = paracetamol.ecfp4()

print(f"Aspirin vs Ibuprofen  : {chematic.tanimoto(fp_a, fp_i):.3f}")
print(f"Aspirin vs Paracetamol: {chematic.tanimoto(fp_a, fp_p):.3f}")
print(f"Ibuprofen vs Paracetamol: {chematic.tanimoto(fp_i, fp_p):.3f}")

# numpy (2048-bit) for scikit-learn
import numpy as np
fp_numpy = aspirin.ecfp4_numpy()  # shape (2048,), dtype uint8
print(f"\nnumpy shape: {fp_numpy.shape}, set bits: {fp_numpy.sum()}")

## 4. Bulk parallel processing

In [ ]:
import pandas as pd

smiles_list = [
    "CC(=O)Oc1ccccc1C(=O)O",         # aspirin
    "CC(C)Cc1ccc(CC(C)C(=O)O)cc1",   # ibuprofen
    "CC(=O)Nc1ccc(O)cc1",            # paracetamol
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C", # caffeine
    "c1ccc2ccccc2c1",                # naphthalene
    "OCC1OC(O)C(O)C(O)C1O",         # glucose
]

# Parallelized across CPU cores automatically
df = pd.DataFrame(chematic.bulk.descriptors(smiles_list))
print(df[["mw", "logp", "tpsa", "qed", "lipinski_passes", "pains_passes"]].round(2))

## 5. Substructure search with SMARTS

In [ ]:
mol = chematic.from_smiles("CC(=O)Oc1ccccc1C(=O)O")  # aspirin

# Boolean match
if chematic.smarts_match("[CX3](=O)[OX2H1]", mol):
    print("carboxylic acid found")

if chematic.smarts_match("c1ccccc1", mol):
    print("benzene ring found")

# Get matched atom indices
matches = chematic.smarts_find("[CX3](=O)[OX2H1]", mol)
print(f"Carboxyl atom indices: {matches}")

# Multiple molecules
mols = [chematic.from_smiles(s) for s in smiles_list]
has_ring = [chematic.smarts_match("c1ccccc1", m) for m in mols]
print(f"\nAromatic ring present: {has_ring}")

## 6. Molecular transformations

In [ ]:
# Standardisation (salt removal, charge neutralisation, tautomer)
salt = chematic.from_smiles("[Na+].[O-]c1ccccc1")
clean = salt.standardize()
print(f"Before: {salt.smiles}")
print(f"After : {clean.smiles}")

# Murcko scaffold
aspirin = chematic.from_smiles("CC(=O)Oc1ccccc1C(=O)O")
scaffold = aspirin.scaffold()
print(f"\nAspirin scaffold: {scaffold.smiles}")

# Tautomers
mol = chematic.from_smiles("OC1=CC=CC=N1")  # 2-pyridinol
canonical = mol.canonical_tautomer()
print(f"\n2-pyridinol canonical tautomer: {canonical.smiles}")
print(f"All tautomers ({len(mol.enumerate_tautomers())} total): {[t.smiles for t in mol.enumerate_tautomers()]}")

# SMIRKS reaction
phenol = chematic.from_smiles("c1ccccc1O")
products = chematic.run_smirks("[OH:1]>>[O-:1]", [phenol])
print(f"\nDeprotonated phenol: {products[0][0].smiles}")

## 7. 2D visualization in Jupyter

In [ ]:
from IPython.display import SVG, display

mol = chematic.from_smiles("CC(=O)Oc1ccccc1C(=O)O")  # aspirin

# Single molecule
display(SVG(mol.svg()))

In [ ]:
# Highlight atoms matched by a SMARTS pattern
matches = chematic.smarts_find("[CX3](=O)[OX2H1]", mol)
atoms = [i for match in matches for i in match]
display(SVG(mol.svg_highlighted(atoms, color="#FF6B6B")))

In [ ]:
# Grid view
mols = [chematic.from_smiles(s) for s in smiles_list[:4]]
display(SVG(chematic.depict_grid(mols, cols=2)))

## 8. 3D shape-based virtual screening (USR)

In [ ]:
# Query: aspirin
query = chematic.from_smiles("CC(=O)Oc1ccccc1C(=O)O")

library = [
    "CC(=O)Nc1ccc(O)cc1",            # paracetamol
    "CC(C)Cc1ccc(CC(C)C(=O)O)cc1",   # ibuprofen
    "c1ccccc1",                       # benzene
    "OCC1OC(O)C(O)C(O)C1O",          # glucose
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C", # caffeine
]

hits = chematic.shape_screen(query, library)
print("Rank  Similarity  SMILES")
print("-" * 60)
for rank, (idx, sim) in enumerate(hits, 1):
    print(f"{rank:4d}  {sim:.3f}       {library[idx]}")

## 9. MCS and InChI

In [ ]:
mol1 = chematic.from_smiles("CC(=O)Oc1ccccc1C(=O)O")  # aspirin
mol2 = chematic.from_smiles("CC(=O)Nc1ccc(O)cc1")     # paracetamol

mcs = chematic.find_mcs([mol1, mol2])
if mcs:
    print(f"MCS: {mcs.smiles}")

# InChI round-trip
mol = chematic.from_smiles("CCO")  # ethanol
print(f"\nInChI   : {mol.inchi}")
print(f"InChIKey: {mol.inchikey}")
mol2 = chematic.from_inchi(mol.inchi)
print(f"Round-trip OK: {mol.formula == mol2.formula}")

---

## What's next?

- [Full cookbook](https://kent-tokyo.github.io/chematic/cookbook/) — 20 copy-paste-ready tasks
- [RDKit migration guide](https://kent-tokyo.github.io/chematic/rdkit_cheatsheet/) — side-by-side API comparison
- [API reference](https://kent-tokyo.github.io/chematic/api/chematic/) — complete Python API
- [GitHub](https://github.com/kent-tokyo/chematic) — source code and issues
- [PyPI](https://pypi.org/project/chematic/) — `pip install chematic`